# Minnie 1822 CCM ETL

Save the Minnie65 materialization 1822 catalog in CCM format.

In [1]:
import gc
import json
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import polars as pl
from deltalake import write_deltalake

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import (
    cell_cell_connectivity_to_arrow,
    derive_cell_cell_connectivity,
    read_cell_cell_connectivity,
    write_models,
)
from connects_common_connectivity.io.arrow_utils import (
    attach_linkml_metadata,
    build_arrow_schema,
    build_cell_feature_matrix_schema,
)
from connects_common_connectivity.io.path_spec import (
    MODEL_TABLE_PATHS,
    WIDE_PAYLOAD_PATHS,
)
from connects_common_connectivity.models import (
    AlgorithmRun,
    AnatomicalOrientation,
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    Cluster,
    ClusterHierarchy,
    ClusterMembership,
    DataItem,
    DataItemDataSetAssociation,
    DataSet,
    Default2DView,
    HierarchyCategory,
    LocationType,
    Modality,
    ReferenceSpace,
    SignedAxis,
    SpatialLocation,
    SynapseConnectivityLong,
    SynapseFeatureMatrix,
    Unit,
)

In [2]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = REPO_ROOT / "data/EM minnie 1822"
OUTPUT_ROOT = Path(get_settings().output_root)

PROJECT_ID = "minnie65"
DATASET_SINGLE_SOMA_NEURONS = "minnie65_v1822_single_soma_neurons"
DATASET_PROOFREAD_AXONS = "minnie65_v1822_proofread_axons"
CAVE_REFERENCE_SPACE_ID = "minnie65_v1822_cave_nm"
STANDARD_REFERENCE_SPACE_ID = "minnie65_v1822_standard_transform_um"
VOLUME_FEATURE_SET_ID = "minnie65_v1822_nucleus_volume"
HIERARCHY_ID = "minnie65_v1822_cell_types"
RUN_ID = "minnie65_v1822_cell_type_annotation"
ROOT_CLUSTER_ID = "neuron"
SYNAPSE_TABLE_ID = "minnie65_v1822_proofread_axon_to_single_soma"
SYNAPSE_FEATURE_MATRIX_ID = "minnie65_v1822_proofread_axon_to_single_soma_features"
CONNECTOME_ID = "minnie65_v1822_proofread_axon_to_single_soma"
PUBLICATION = "https://www.microns-explorer.org/"

SYNAPSE_PATH = DATA_ROOT / "minnie_v1822_proofread_axon_synapses.parquet"
assert SYNAPSE_PATH.is_file(), f"Missing local input: {SYNAPSE_PATH}"

In [4]:
positions = pd.read_parquet(DATA_ROOT / "minnie_v1822_nucleus_positions.parquet")
volumes = pd.read_parquet(DATA_ROOT / "minnie_v1822_nucleus_volume.parquet")
proofreading = pd.read_parquet(DATA_ROOT / "minnie_v1822_proofreading_status.parquet")
cell_types = pd.read_parquet(DATA_ROOT / "minnie_v1822_cell_types.parquet")
neuron_nuclei = pd.read_parquet(DATA_ROOT / "minnie_v1822_neuron_nuclei.parquet")

# Validate source keys before using root ids as CCM DataItem identities.
position_columns = {
    "id", "pt_root_id", "pt_position_x", "pt_position_y", "pt_position_z",
    "x_medial-lateral", "y_dorsal-ventral", "z_caudal-rostral",
}
assert position_columns <= set(positions.columns)
assert positions["id"].is_unique
assert positions["pt_root_id"].is_unique
assert (positions["pt_root_id"] != 0).all()
assert neuron_nuclei["cell_type"].eq("neuron").all()

# The upstream CAVE dedup already guarantees one soma per root id, so keeping only
# nuclei classified as neurons leaves exactly the single-soma neuron cohort.
positions = positions.loc[
    positions["id"].isin(set(neuron_nuclei["id"]))
].reset_index(drop=True)

# CCM identifiers are strings even when the source stores numeric root ids.
positions["dataitem_id"] = positions["pt_root_id"].astype(str)
registered_ids = set(positions["dataitem_id"])
print(f"single soma neurons: {len(registered_ids):,}")

single soma neurons: 81,059


## dataset dataitem

In [4]:
datasets = [
    DataSet(
        id=DATASET_SINGLE_SOMA_NEURONS,
        name="Minnie65 materialization 1822 single soma neurons",
        publication=PUBLICATION,
        modality=Modality.ELECTRON_MICROSCOPY,
        project_id=PROJECT_ID,
    ),
    DataSet(
        id=DATASET_PROOFREAD_AXONS,
        name="Minnie65 materialization 1822 proofread axons",
        publication=PUBLICATION,
        modality=Modality.ELECTRON_MICROSCOPY,
        project_id=PROJECT_ID,
    ),
]
print(write_models(datasets, output_root=OUTPUT_ROOT))

dataitems = [
    DataItem(id=row.dataitem_id, name=str(row.id), project_id=PROJECT_ID)
    for row in positions.itertuples()
]
print(write_models(dataitems, output_root=OUTPUT_ROOT))

WrittenResult(class_name='DataSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/dataset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'minnie65\')',), rows_written=2)
WrittenResult(class_name='DataItem', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/dataitem'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'minnie65\')',), rows_written=81059)


In [6]:
# Stale assessments are excluded so cohort membership reflects current proofreader status.
fresh_proofreading = proofreading.loc[~proofreading["assessment_stale"]].copy()
proofread_axon_ids = set(
    fresh_proofreading.loc[
        fresh_proofreading["status_axon"]
        & fresh_proofreading["strategy_axon"].eq("axon_fully_extended"),
        "pt_root_id",
    ].astype(str)
) & registered_ids

# Associations represent cohort membership without duplicating DataItem records.
associations = [
    DataItemDataSetAssociation(
        dataitem_id=dataitem_id,
        dataset_id=DATASET_SINGLE_SOMA_NEURONS,
        project_id=PROJECT_ID,
    )
    for dataitem_id in sorted(registered_ids)
]
associations.extend(
    DataItemDataSetAssociation(
        dataitem_id=dataitem_id,
        dataset_id=DATASET_PROOFREAD_AXONS,
        project_id=PROJECT_ID,
    )
    for dataitem_id in sorted(proofread_axon_ids)
)
assert {row.dataitem_id for row in associations} <= registered_ids
print(write_models(associations, output_root=OUTPUT_ROOT))
print(f"single soma neurons: {len(registered_ids):,}")
print(f"proofread axons: {len(proofread_axon_ids):,}")

WrittenResult(class_name='DataItemDataSetAssociation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/dataitem_dataset_association'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataset_id" = source."dataset_id" AND target."dataitem_id" = source."dataitem_id" AND target."project_id" IN (\'minnie65\')',), rows_written=0)
single soma neurons: 81,059
proofread axons: 627


## spatial location

The cached standard-transform coordinates must be in microns, as returned by `minnie_transform_nm()` without further scaling. Regenerate legacy millimetre-valued caches with the corrected CAVE query notebook before running this ETL. For a spatial-only repair of an existing release, use [fix_minnie_spatial.ipynb](fix_minnie_spatial.ipynb).

Keep the existing reference-space IDs: `write_models` merges spatial rows on project, data item, reference space, and location type, updating coordinates rather than appending duplicate rows.

In [8]:
default_view = Default2DView(
    left_to_right=SignedAxis.PLUS_X,
    bottom_to_top=SignedAxis.MINUS_Y,
)
reference_spaces = [
    ReferenceSpace(
        id=CAVE_REFERENCE_SPACE_ID,
        name="Minnie65 materialization 1822 CAVE coordinates",
        description="CAVE nucleus positions requested at 1 nm resolution.",
        unit=Unit.NANOMETERS_LENGTH,
        default_2d_view=default_view,
        project_id=PROJECT_ID,
    ),
    ReferenceSpace(
        id=STANDARD_REFERENCE_SPACE_ID,
        name="Minnie65 standard-transform coordinates",
        description="Minnie standard-transform coordinates in microns.",
        unit=Unit.MICRONS_LENGTH,
        default_2d_view=default_view,
        anatomical_orientation=AnatomicalOrientation(
            medial_to_lateral=SignedAxis.PLUS_X,
            dorsal_to_ventral=SignedAxis.PLUS_Y,
            rostral_to_caudal=SignedAxis.MINUS_Z,
            pia_to_white_matter=SignedAxis.PLUS_Y,
        ),
        project_id=PROJECT_ID,
    ),
]
print(write_models(reference_spaces, output_root=OUTPUT_ROOT))

spatial_locations = []
for row in positions.to_dict("records"):
    common = {
        "dataitem_id": row["dataitem_id"],
        "location_type": LocationType.SOMA,
        "project_id": PROJECT_ID,
    }
    spatial_locations.extend(
        [
            SpatialLocation(
                reference_space=CAVE_REFERENCE_SPACE_ID,
                x=float(row["pt_position_x"]),
                y=float(row["pt_position_y"]),
                z=float(row["pt_position_z"]),
                **common,
            ),
            SpatialLocation(
                reference_space=STANDARD_REFERENCE_SPACE_ID,
                x=float(row["x_medial-lateral"]),
                y=float(row["y_dorsal-ventral"]),
                z=float(row["z_caudal-rostral"]),
                **common,
            ),
        ]
    )
print(write_models(spatial_locations, output_root=OUTPUT_ROOT))

WrittenResult(class_name='ReferenceSpace', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/referencespace'), mode='merge_scoped', predicates=('(target."project_id" IS NOT DISTINCT FROM source."project_id") AND target."id" = source."id"',), rows_written=2)
WrittenResult(class_name='SpatialLocation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/spatiallocation'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataitem_id" = source."dataitem_id" AND target."reference_space" = source."reference_space" AND target."location_type" = source."location_type" AND target."project_id" IN (\'minnie65\')',), rows_written=162118)


## features

In [9]:
# Join on both source keys to prevent a volume from being attached to the wrong root.
volume_frame = positions[["id", "pt_root_id", "dataitem_id"]].merge(
    volumes, on=["id", "pt_root_id"], how="left", validate="one_to_one"
)
assert volume_frame["volume"].notna().all()
assert set(volume_frame["dataitem_id"]) == registered_ids

# Definitions describe columns; the feature set groups them into one matrix contract.
volume_definition = CellFeatureDefinition(
    id="nucleus_volume_um",
    description="Nucleus volume from the CAVE nucleus detection table.",
    unit=Unit.MICRONS_CUBED.value,
    data_type="<f4",
    range_min=0.0,
    project_id=PROJECT_ID,
    feature_set_id=VOLUME_FEATURE_SET_ID,
)
volume_feature_set = CellFeatureSet(
    id=VOLUME_FEATURE_SET_ID,
    description="Minnie65 materialization 1822 nucleus volume.",
    feature_definition_ids=[volume_definition.id],
    extraction_method="CAVE nucleus_detection_v0 at materialization 1822",
    project_id=PROJECT_ID,
)
print(write_models([volume_definition], output_root=OUTPUT_ROOT))
print(write_models([volume_feature_set], output_root=OUTPUT_ROOT))

WrittenResult(class_name='CellFeatureDefinition', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/cellfeaturedefinition'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."feature_set_id" = source."feature_set_id" AND target."id" = source."id" AND target."project_id" IN (\'minnie65\') AND target."feature_set_id" IN (\'minnie65_v1822_nucleus_volume\')',), rows_written=1)
WrittenResult(class_name='CellFeatureSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/cellfeatureset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'minnie65\')',), rows_written=1)


In [10]:
volume_wide = pd.DataFrame(
    {
        "id": volume_frame["dataitem_id"],
        "project_id": PROJECT_ID,
        "feature_set_id": VOLUME_FEATURE_SET_ID,
        volume_definition.id: volume_frame["volume"].astype("float32"),
    }
)
volume_schema = build_cell_feature_matrix_schema(
    volume_feature_set, [volume_definition], cell_index_column="id"
)
volume_table = pa.Table.from_pandas(
    volume_wide, schema=volume_schema, preserve_index=False
)
volume_path = OUTPUT_ROOT / f"cellfeatures/{VOLUME_FEATURE_SET_ID}"
write_deltalake(
    volume_path,
    volume_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
volume_matrix = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{VOLUME_FEATURE_SET_ID}",
    feature_set_id=VOLUME_FEATURE_SET_ID,
    parquet_path=volume_path.relative_to(OUTPUT_ROOT).as_posix(),
    cell_index_column="id",
    project_id=PROJECT_ID,
)
print(write_models([volume_matrix], output_root=OUTPUT_ROOT))

WrittenResult(class_name='CellFeatureMatrix', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/cellfeaturematrix'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."feature_set_id" = source."feature_set_id" AND target."id" = source."id" AND target."project_id" IN (\'minnie65\')',), rows_written=1)


## cell types

In [22]:
# Normalize the two source classification systems into stable CCM class nodes.
class_by_system = {"excitatory": "exc", "inhibitory": "inh"}
unknown_systems = set(cell_types["classification_system"].unique()) - set(class_by_system)
assert not unknown_systems, f"Unmapped classification systems: {sorted(unknown_systems)}"

# cell_type_multifeature_combo keeps one row per nucleus, so a merged segment can
# appear more than once; drop every ambiguous root rather than picking one.
typed_source = cell_types.loc[cell_types["pt_root_id"] != 0]
typed_source = typed_source.drop_duplicates(subset=["pt_root_id"], keep=False)

typed_cells = typed_source.merge(
    positions[["pt_root_id", "dataitem_id"]],
    on="pt_root_id",
    how="inner",
    validate="one_to_one",
).copy()
typed_cells["class"] = typed_cells["classification_system"].map(class_by_system)
assert typed_cells["dataitem_id"].is_unique
assert set(typed_cells["dataitem_id"]) <= registered_ids

# Each source leaf label must have one parent and remain unique among hierarchy nodes.
leaf_parents = typed_cells.groupby("cell_type")["class"].unique()
assert all(len(parents) == 1 for parents in leaf_parents), (
    "A source cell_type maps to multiple classes."
)
leaf_to_class = {label: parents[0] for label, parents in leaf_parents.items()}
reserved_cluster_ids = {ROOT_CLUSTER_ID, *class_by_system.values()}
assert set(leaf_to_class).isdisjoint(reserved_cluster_ids)

# Reuse historical MICrONS colors where available; otherwise inherit class color.
class_colors = {"neuron": "#000000", "exc": "#FF0000", "inh": "#0000FF"}
cell_type_rgb = {
    "L2IT": [0.9075666881074735, 0.7831311441799196, 0.6949858653492867],
    "L3IT": [0.8708834148859907, 0.6888410680873137, 0.6022311567741945],
    "L4IT": [0.8274067443395999, 0.5906550752430283, 0.5233409192850811],
    "L5ET": [0.7795806757764551, 0.5022849626842287, 0.4643377130216367],
    "L5IT": [0.719621740596695, 0.41459615740890743, 0.41383679416754976],
    "L5NP": [0.6539490167887168, 0.3392197573102502, 0.37390086854124416],
    "L6CT": [0.5747615524284997, 0.26764431745066686, 0.3354259029195752],
    "L6IT": [0.4927058890394963, 0.20865645456553394, 0.299493097642672],
    "DTC": [0.5054276262176057, 0.7742578554634272, 0.7550447895194092],
    "ITC": [0.307398668309227, 0.535864603420623, 0.6514437955029915],
    "PV": [47 / 255, 107 / 255, 85 / 255],
    "MC": [62 / 255, 122 / 255, 92 / 255],
    "NGC": [86 / 255, 143 / 255, 105 / 255],
    "L1": [107 / 255, 155 / 255, 120 / 255],
    "NMC": [71 / 255, 112 / 255, 95 / 255],
    "AltDTC": [102 / 255, 138 / 255, 108 / 255],
    "AltBasket": [49 / 255, 95 / 255, 73 / 255],
    "ITCperi": [84 / 255, 123 / 255, 99 / 255],
    "ChC": [120 / 255, 157 / 255, 130 / 255],
}
cell_type_colors = {
    label: (
        "#{:02x}{:02x}{:02x}".format(*(int(channel * 255) for channel in cell_type_rgb[label]))
        if label in cell_type_rgb
        else class_colors[leaf_to_class[label]]
    )
    for label in leaf_to_class
}
assert set(cell_type_colors) == set(leaf_to_class)

In [23]:
class_ids = sorted(set(class_by_system.values()))
leaf_labels = sorted(leaf_to_class)
clusters = [
    Cluster(
        id=ROOT_CLUSTER_ID,
        hierarchy_id=HIERARCHY_ID,
        children=class_ids,
        level=0,
        hex_color=class_colors[ROOT_CLUSTER_ID],
    )
]
clusters.extend(
    Cluster(
        id=class_id,
        hierarchy_id=HIERARCHY_ID,
        parent=ROOT_CLUSTER_ID,
        children=[
            label
            for label in leaf_labels
            if leaf_to_class[label] == class_id
        ],
        level=1,
        hex_color=class_colors[class_id],
        hierarchy_category="Class",
    )
    for class_id in class_ids
)
clusters.extend(
    Cluster(
        id=label,
        hierarchy_id=HIERARCHY_ID,
        parent=leaf_to_class[label],
        level=2,
        hex_color=cell_type_colors[label],
        hierarchy_category="Cluster",
    )
    for label in leaf_labels
)
assert all(cluster.hex_color is not None for cluster in clusters)
print(write_models(clusters, output_root=OUTPUT_ROOT))

WrittenResult(class_name='Cluster', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/cluster'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'minnie65_v1822_cell_types\')',), rows_written=9)


In [20]:
run = AlgorithmRun(
    id=RUN_ID,
    algorithm_name="CAVE cell-type annotation",
    algorithm_version="minnie65 materialization 1822",
    input_dataset=DATASET_SINGLE_SOMA_NEURONS,
    json_object=json.dumps(
        {
            "source": "cell_type_multifeature_combo",
            "classification_column": "classification_system",
            "cluster_column": "cell_type",
        }
    ),
)
categories = [
    HierarchyCategory(
        id="Class",
        hierarchy_id=HIERARCHY_ID,
        description="Excitatory or inhibitory class.",
        level=1,
    ),
    HierarchyCategory(
        id="Cluster",
        hierarchy_id=HIERARCHY_ID,
        description="Exact source cell_type label.",
        level=2,
    ),
]
hierarchy = ClusterHierarchy(
    id=HIERARCHY_ID,
    run=RUN_ID,
    root=ROOT_CLUSTER_ID,
    clusters=[cluster.id for cluster in clusters],
)
print(write_models([run], output_root=OUTPUT_ROOT))
print(write_models(categories, output_root=OUTPUT_ROOT))
print(write_models([hierarchy], output_root=OUTPUT_ROOT))

WrittenResult(class_name='AlgorithmRun', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/algorithmrun'), mode='merge_scoped', predicates=('target."id" = source."id"',), rows_written=0)
WrittenResult(class_name='HierarchyCategory', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/hierarchycategory'), mode='merge_scoped', predicates=('target."hierarchy_id" = source."hierarchy_id" AND target."id" = source."id" AND target."hierarchy_id" IN (\'minnie65_v1822_cell_types\')',), rows_written=0)
WrittenResult(class_name='ClusterHierarchy', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/clusterhierarchy'), mode='merge_scoped', predicates=('target."id" = source."id"',), rows_written=1)


In [21]:
memberships = []
for row in typed_cells[["dataitem_id", "class", "cell_type"]].to_dict("records"):
    assigned = [
        ROOT_CLUSTER_ID,
        row["class"],
        row["cell_type"],
    ]
    memberships.extend(
        ClusterMembership(
            item=row["dataitem_id"],
            cluster=cluster_id,
            hierarchy_id=HIERARCHY_ID,
            probability=1.0,
            project_id=PROJECT_ID,
        )
        for cluster_id in assigned
    )
assert len(memberships) == 3 * len(typed_cells)
print(write_models(memberships, output_root=OUTPUT_ROOT))

WrittenResult(class_name='ClusterMembership', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/clustermembership'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."hierarchy_id" = source."hierarchy_id" AND target."item" = source."item" AND target."cluster" = source."cluster" AND target."project_id" IN (\'minnie65\') AND target."hierarchy_id" IN (\'minnie65_v1822_cell_types\')',), rows_written=207906)


## synapses

In [ ]:
# Free completed object batches before loading the multi-million-row synapse table.
# CCM bulk tables use columnar Arrow writes instead of one model object per row.
del spatial_locations, associations, dataitems, memberships, volume_wide, volume_table
gc.collect()

NameError: name 'spatial_locations' is not defined

In [34]:
synapse_source = pq.read_table(SYNAPSE_PATH)

synapse_source = synapse_source.filter(
    pc.and_(
        pc.is_in(
            synapse_source["pre_pt_root_id"],
            value_set=pa.array(
                [int(value) for value in proofread_axon_ids],
                type=synapse_source["pre_pt_root_id"].type,
            ),
        ),
        pc.is_in(
            synapse_source["post_pt_root_id"],
            value_set=pa.array(
                [int(value) for value in registered_ids],
                type=synapse_source["post_pt_root_id"].type,
            ),
        ),
    )
)

synapse_count = synapse_source.num_rows
assert pc.count_distinct(synapse_source["id"]).as_py() == synapse_count
print(f"proofread axon to single-soma synapses: {synapse_count:,}")


proofread axon to single-soma synapses: 1,079,400


In [35]:
# The long table stores only synapse identity and endpoints. Measurements such as
# size and position belong in a wide feature payload keyed by the same synapse id.
long_schema = build_arrow_schema(SynapseConnectivityLong)
long_table = pa.Table.from_arrays(
    [
        pa.repeat(PROJECT_ID, synapse_count),
        pc.cast(synapse_source["id"], pa.string()),
        pc.cast(synapse_source["pre_pt_root_id"], pa.string()),
        pc.cast(synapse_source["post_pt_root_id"], pa.string()),
        pa.repeat(SYNAPSE_TABLE_ID, synapse_count),
    ],
    schema=long_schema,
)
long_table = attach_linkml_metadata(
    long_table,
    linkml_class="SynapseConnectivityLong",
)
write_deltalake(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["SynapseConnectivityLong"],
    long_table,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"synapse_table_id = '{SYNAPSE_TABLE_ID}'"
    ),
    partition_by=["project_id"],
)
del long_table
gc.collect()

# Endpoint root ids already live in the long table, so the feature matrix keeps
# only the source measurements and adds the CCM scope columns used for filtering.
feature_columns = [
    column
    for column in synapse_source.column_names
    if column != "id" and not column.endswith("_root_id")
]
synapse_wide = synapse_source.select(["id", *feature_columns]).set_column(
    0,
    "id",
    pc.cast(synapse_source["id"], pa.string()),
)
synapse_wide = synapse_wide.append_column(
    "project_id",
    pa.repeat(PROJECT_ID, synapse_count),
)
synapse_wide = synapse_wide.append_column(
    "synapse_table_id",
    pa.repeat(SYNAPSE_TABLE_ID, synapse_count),
)
synapse_feature_path = (
    OUTPUT_ROOT
    / WIDE_PAYLOAD_PATHS["synapse_features"]
    / SYNAPSE_FEATURE_MATRIX_ID
)
write_deltalake(
    synapse_feature_path,
    synapse_wide,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"synapse_table_id = '{SYNAPSE_TABLE_ID}'"
    ),
    partition_by=["project_id"],
)
synapse_feature_matrix = SynapseFeatureMatrix(
    id=SYNAPSE_FEATURE_MATRIX_ID,
    description=(
        "Per-synapse size and center position for Minnie65 materialization 1822."
    ),
    synapse_table_id=SYNAPSE_TABLE_ID,
    project_id=PROJECT_ID,
    parquet_path=synapse_feature_path.relative_to(OUTPUT_ROOT).as_posix(),
    synapse_index_column="id",
)
print(write_models([synapse_feature_matrix], output_root=OUTPUT_ROOT))
print(f"synapses: {synapse_count:,}")
del synapse_source, synapse_wide
gc.collect()

WrittenResult(class_name='SynapseFeatureMatrix', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v1/synapsefeaturematrix'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'minnie65\')',), rows_written=1)
synapses: 1,079,400


20

# connectivity

In [36]:
# Read only aggregation columns, then keep synapses from a proofread axon onto a
# registered single-soma neuron before creating pair-level measurements.
connectivity_source = pl.read_parquet(
    SYNAPSE_PATH,
    columns=["id", "pre_pt_root_id", "post_pt_root_id", "size"],
)
selected_synapses = (
    connectivity_source
    .filter(
        pl.col("pre_pt_root_id").is_in([int(value) for value in proofread_axon_ids])
        & pl.col("post_pt_root_id").is_in([int(value) for value in registered_ids])
    )
    .select(
        pl.col("id").cast(pl.String),
        pl.col("pre_pt_root_id").cast(pl.String).alias("presynaptic_cell"),
        pl.col("post_pt_root_id").cast(pl.String).alias("postsynaptic_cell"),
        pl.col("size"),
        pl.lit(PROJECT_ID).alias("project_id"),
        pl.lit(SYNAPSE_TABLE_ID).alias("synapse_table_id"),
    )
)
pair_count = selected_synapses.select(
    "presynaptic_cell",
    "postsynaptic_cell",
).unique().height

# Each connected pair gets one count row and one summed-size row. The source
# synapse table id is retained as provenance but is not part of pair grouping.
connectivity = derive_cell_cell_connectivity(
    selected_synapses,
    project_id=PROJECT_ID,
    connectome_id=CONNECTOME_ID,
    modality=Modality.ELECTRON_MICROSCOPY,
    size_column="size",
    size_unit=Unit.ARBITRARY_UNIT,
)
connectivity_table = attach_linkml_metadata(
    cell_cell_connectivity_to_arrow(connectivity),
    linkml_class="CellCellConnectivityLong",
)
write_deltalake(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["CellCellConnectivityLong"],
    connectivity_table,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"connectome_id = '{CONNECTOME_ID}'"
    ),
    partition_by=["project_id", "connectome_id", "measurement_type"],
)
print(
    f"connectivity: {pair_count:,} pairs | "
    f"{connectivity_table.num_rows:,} measurements"
)
del connectivity_source, selected_synapses, connectivity, connectivity_table
gc.collect()

connectivity: 590,180 pairs | 1,180,360 measurements


10

## verify

In [ ]:
# Read stored model tables back by project or hierarchy scope rather than relying
# on notebook variables to prove that CCM relationships were persisted correctly.
stored_associations = pl.read_delta(OUTPUT_ROOT / "dataitem_dataset_association").filter(
    (pl.col("project_id") == PROJECT_ID)
    & pl.col("dataset_id").is_in(
        [DATASET_SINGLE_SOMA_NEURONS, DATASET_PROOFREAD_AXONS]
    )
)
stored_clusters = pl.read_delta(OUTPUT_ROOT / "cluster").filter(
    pl.col("hierarchy_id") == HIERARCHY_ID
)
stored_memberships = pl.read_delta(OUTPUT_ROOT / "clustermembership").filter(
    (pl.col("project_id") == PROJECT_ID)
    & (pl.col("hierarchy_id") == HIERARCHY_ID)
)
expected_cluster_colors = {
    ROOT_CLUSTER_ID: class_colors[ROOT_CLUSTER_ID],
    **{class_id: class_colors[class_id] for class_id in class_ids},
    **{label: cell_type_colors[label] for label in leaf_labels},
}
stored_cluster_colors = dict(
    stored_clusters.select("id", "hex_color").iter_rows()
)
association_counts = dict(
    stored_associations.group_by("dataset_id").len().iter_rows()
)
assert association_counts == {
    DATASET_SINGLE_SOMA_NEURONS: len(registered_ids),
    DATASET_PROOFREAD_AXONS: len(proofread_axon_ids),
}
assert stored_clusters.height == len(clusters)
assert stored_cluster_colors == expected_cluster_colors
assert stored_memberships.height == 3 * len(typed_cells)
assert set(stored_memberships["item"].to_list()) <= registered_ids
assert set(volume_frame["dataitem_id"]) == registered_ids

# Stream row counts for large payloads so verification does not load all synapses.
stored_synapse_count = (
    pl.scan_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["SynapseConnectivityLong"])
    .filter(
        (pl.col("project_id") == PROJECT_ID)
        & (pl.col("synapse_table_id") == SYNAPSE_TABLE_ID)
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)
stored_synapse_feature_count = (
    pl.scan_delta(synapse_feature_path)
    .filter(
        (pl.col("project_id") == PROJECT_ID)
        & (pl.col("synapse_table_id") == SYNAPSE_TABLE_ID)
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)
stored_connectivity = read_cell_cell_connectivity(
    PROJECT_ID,
    CONNECTOME_ID,
    synapse_table_id=SYNAPSE_TABLE_ID,
    output_root=OUTPUT_ROOT,
)

# Identity and feature tables stay row-aligned, while each connected pair has
# exactly the count and summed-size measurements emitted by the helper.
assert stored_synapse_count == synapse_count
assert stored_synapse_feature_count == synapse_count
assert stored_connectivity.height == pair_count * 2
assert stored_connectivity["measurement_type"].n_unique() == 2
assert set(stored_connectivity["presynaptic_cell"]) <= proofread_axon_ids
assert set(stored_connectivity["postsynaptic_cell"]) <= registered_ids
print(
    "Minnie 1822 datasets, spatial locations, features, cell types, "
    "synapses, and connectivity verified."
)

Minnie 1822 datasets, spatial locations, features, cell types, synapses, and connectivity verified.
